In [1]:
import altair as alt
import pandas as pd
from omegaconf import OmegaConf

from update_vars import GTFS_DATA_DICT, RT_SCHED_GCS, SCHED_GCS, SEGMENT_GCS

In [2]:
readable_dict = OmegaConf.load("readable2.yml")

In [3]:
subset = [
    "direction_id",
    "time_period",
    "avg_scheduled_service_minutes",
    "n_scheduled_trips",
    "service_date",
    "recent_combined_name",
    "route_primary_direction",
    "minutes_atleast1_vp",
    "minutes_atleast2_vp",
    "is_early",
    "is_ontime",
    "is_late",
    "vp_per_minute",
    "pct_in_shape",
    "pct_sched_journey_atleast1_vp",
    "pct_sched_journey_atleast2_vp",
    "rt_sched_journey_ratio",
    "speed_mph",
    "portfolio_organization_name",
    "headway_in_minutes",
 "sched_rt_category" # added this
]

In [4]:
readable_col_names = {
    "direction_id": "Direction (0/1)",
    "time_period": "Period",
    "avg_scheduled_service_minutes": "Average Scheduled Service (trip minutes)",
    "n_scheduled_trips": "# Scheduled Trips",
    "service_date": "Date",
    "recent_combined_name": "Route",
    "route_primary_direction": "Direction",
    "minutes_atleast1_vp": "# Minutes with 1+ VP per Minute",
    "minutes_atleast2_vp": "# Minutes with 2+ VP per Minute",
    "is_early": "# Early Arrival Trips",
    "is_ontime": "# On-Time Trips",
    "is_late": "# Late Trips",
    "vp_per_minute": "Average VP per Minute",
    "pct_in_shape": "% VP within Scheduled Shape",
    "pct_sched_journey_atleast1_vp": "% Scheduled Trip w/ 1+ VP/Minute",
    "pct_sched_journey_atleast2_vp": "% Scheduled Trip w/ 2+ VP/Minute",
    "rt_sched_journey_ratio": "Realtime versus Scheduled Service Ratio",
    "speed_mph": "Speed (MPH)",
    "portfolio_organization_name": "Portfolio Organization Name",
    "headway_in_minutes": "Headway (Minutes)",
}

In [5]:
def data_wrangling_for_visualizing2(df):
    """
    Depending on how much this is used, some stuff
    might be moved outside to be variables borrowed elsewhere.
    """
    
    # create new columns
    # what is the formatting on this? it should be included...for now, it's in the floats
    df = df.assign(
        headway_in_minutes = 60 / df.frequency
    )
    
    # these show up as floats but should be integers
    # also these aren't kept...
    route_typology_cols = [
        f"is_{c}" for c in 
        ["express", "rapid",
         "ferry", "rail", "coverage",
         "local", "downtown_local"]
    ]
    
    # the pct_ columns are included here....do you want to round it first
    # and then scale it up? i dealt with this by excluding it
    float_cols = [c for c in df.select_dtypes(include=["float"]).columns 
                     if c not in route_typology_cols and "pct" not in c]
    
    df[float_cols] = df[float_cols].round(2)
    
    # these had 3 decimal places, then when it gets scaled, it just has 1 decimal place
    # is that what you want? or you want it rounded to the nearest integer?
    # whatever you decide, it should be obvious bc of the code, not because 
    # of the order of the code execution
    pct_cols = [c for c in df.columns if "pct" in c]
    df[pct_cols] = df[pct_cols] * 100

    
    # subset to schedule and vp / why is this done now? do you publish schedule operators?
    # or do schedule_only operators only get the first section?
    # the subset columns is missing sched_rt_category, and it needs an
    # entry in the rename dict if it's used within text table as combo column?
    
    df2 = df.assign(
        time_period = df.time_period.str.replace("_", " ").str.title()
    )[subset].query(
        'sched_rt_category == "schedule_and_vp"'
    ).rename(
        columns = readable_col_names
    ).reset_index(drop=True)

    return df2

In [6]:
FILE = GTFS_DATA_DICT.digest_tables.route_schedule_vp

# some of the portfolio grain can be dealt with
# but separate out the renaming/replacing/subsetting to separate script

df = pd.read_parquet(
    f"{RT_SCHED_GCS}{FILE}.parquet",
    filters = [[
        ("portfolio_organization_name", "==", "City of West Hollywood"),
        ("recent_combined_name", "in", ["Cityline Local-East", "Cityline Local-West"])
    ]]
).pipe(
    data_wrangling_for_visualizing2
)

In [7]:
import _report_visuals_utils as A1
from IPython.display import HTML, display

# Set drop down menu to be on the upper right for the charts
display(
    HTML(
        """
<style>
form.vega-bindings {
  position: absolute;
  right: 0px;
  top: 0px;
}
</style>
"""
    )
)

In [8]:
specific_chart_dict = readable_dict.spatial_accuracy_graph
specific_chart_dict

{'title': 'Spatial Accuracy', 'subtitle': "The percentage of vehicle positions that fall within 35 meters of a route's scheduled shape (path) reflects the accuracy of the collected spatial data.", 'tooltip': ['Date', 'Route', 'Direction', '% VP within Scheduled Shape'], 'domain': [0, 20, 40, 60, 80, 100], 'color': ['#dd217d', '#fc5c04', '#ff9c42', '#fcb40e', '#e9d868', '#ccbb44']}

In [9]:
def ruler_chart(df: pd.DataFrame, ruler_value: int) -> pd.DataFrame:
    # Add the ruler column
    df = df.assign(
        ruler = ruler_value
    )
    chart = (
        alt.Chart(df)
        .mark_rule(color="red", strokeDash=[10, 7])
        .encode(y="mean(ruler):Q")
    )
    return chart

In [10]:
def bar_chart(
    x_col: str,
    y_col: str,
    color_col: str,
    color_scheme: list,
    tooltip_cols: list,
    date_format: str = "%b %Y",
) -> alt.Chart:

    chart = (
        alt.Chart()
        .mark_bar()
        .encode(
            x=alt.X(
                x_col,
                title=x_col,
                axis=alt.Axis(labelAngle=-45, format=date_format),
            ),
            y=alt.Y(y_col, title=y_col),
            color=alt.Color(
                color_col,
                legend=None,
                title=color_col,
                scale=alt.Scale(range=color_scheme),
            ),
            tooltip=tooltip_cols,
        )
    )

    return chart

In [11]:
def sample_spatial_accuracy_chart(df):
    
    ruler = ruler_chart(df, 100)

    bar = bar_chart(
        x_col = "Date", 
        y_col = "% VP within Scheduled Shape", 
        color_col = "% VP within Scheduled Shape", 
        color_scheme = [*specific_chart_dict.color], 
        tooltip_cols = [*specific_chart_dict.tooltip], 
        date_format="%Y-Q%q"
    )
   
    chart = alt.layer(bar, ruler, data = df).properties(width=200, height=250)
    chart = chart.facet(
        column=alt.Column(
            "Direction:N",
        )
    ).properties(
        title={
            "text": specific_chart_dict.title,
            "subtitle": specific_chart_dict.subtitle,
        }
    )
    
    return chart

In [12]:
def route_filter(df):
    routes_list = df["Route"].unique().tolist()

    route_dropdown = alt.binding_select(
        options=routes_list,
        name="Routes: ",
    )
    # Column that controls the bar charts
    xcol_param = alt.selection_point(
        fields=["Route"], value=routes_list[0], bind=route_dropdown
    )
    
    #all_day = df[df.Period=="All Day"]
    
    chart1 = (
        sample_spatial_accuracy_chart(df[df.Period=="All Day"])
        .add_params(xcol_param)
        .transform_filter(xcol_param)
    ) 

    chart_list = [chart1]
    chart = alt.vconcat(*chart_list)

    
    return chart
 

In [13]:
route_filter(df)

alt.VConcatChart(...)

In [14]:
def old_chart(df):
    df = df.assign(
        ruler = 100
    )
    ruler = (
        alt.Chart(df)
        .mark_rule(color="red", strokeDash=[10, 7])
        .encode(y=f"mean(ruler):Q")
    )
    y_col = "% VP within Scheduled Shape"
    chart = (
            alt.Chart(df)
            .mark_bar(size=7, clip=True)
            .encode(
                x=alt.X(
                    "yearmonthdate(Date):O",
                ),
                y=alt.Y(y_col),
                color=alt.Color(f"{y_col}"),
                tooltip = ["Direction", "Route", y_col]
            )
        )
    
    # All charts
    chart = (chart + ruler).properties(width=200, height=250)
    chart = chart.facet(
        column=alt.Column(
            "Direction:N",
        )
    ).properties(
        title={
            "text": specific_chart_dict.title,
            "subtitle": specific_chart_dict.subtitle,
        }
    )
    
    return chart

In [15]:
def old_route_filter(df):
    routes_list = df["Route"].unique().tolist()

    route_dropdown = alt.binding_select(
        options=routes_list,
        name="Routes: ",
    )
    # Column that controls the bar charts
    xcol_param = alt.selection_point(
        fields=["Route"], value=routes_list[0], bind=route_dropdown
    )
    
    all_day = df[df.Period=="All Day"]
    
    chart1 = (
        old_chart(all_day)
        .add_params(xcol_param)
        .transform_filter(xcol_param)
    ) 

    chart_list = [chart1]
    chart = alt.vconcat(*chart_list)

    
    return chart
 

In [16]:
old_route_filter(df)

alt.VConcatChart(...)